**`harmonize_properties`**

Create a spine for properties (identifiers for all observed properties)

The current version is specific to the Massachusetts, U.S. context (not generalized)

# Configure

In [ ]:
import argparse

# import warnings
# import geopandas as gpd
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
from openplaces.api import get_admin, get_entities
from openplaces.core.schema import AdminId

# from openplaces.geo.ids import (
#     add_openlocationcode_index,
# )
# from openplaces.geo.overlay import overlay_polygons
# from openplaces.geo.polygon import (
#     get_areas,
#     get_intersection_over_union,
#     resolve_overlapping_polygons,
# )
from openplaces.io import save_parquet  # share

# from openplaces.io.transform import make_index_unique, remap
from openplaces.path import path  # external_path, share_path
from openplaces.recipe import find_entity_recipe_id
# from openplaces.recipe import get_output_path
# from openplaces.viz import show_building

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(
    description='Harmonize building data from multiple recipes'
)
parser.add_argument(
    '--recipe_id',
    help='Harmonization recipe (e.g. "US_footprint-cheer-2026")',
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-RI")',
    nargs='*',
)
parser.add_argument(
    '--show_examples',
    help='Make maps of examples (using matplotlib)',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Medford, Middlesex, Massachusetts, United States
    '--admin_ids US-MA-MI --show_examples --verbose'
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

# Prepare data

In [ ]:
# Future start of loop
for admin_id in args.admin_ids:
    break

print(admin_id)

# For visualization purposes
admin4 = get_admin(admin_id, level=4, recipe='US_admin-census-2021_admin4', geom=True)
# admin4

In [ ]:
state_id = str(AdminId(*AdminId(admin_id).levels[:2]))
recipe_id = find_entity_recipe_id(state_id, 'parcel', stage='ingest')
print(recipe_id)

## Prepare properties

In [ ]:
properties = get_entities(recipe_id, admin_id, layer='property')
properties.sample(5).T

## Prepare parcels

In [ ]:
parcels = get_entities(recipe_id, admin_id, geom=True)

In [ ]:
join_id_column = 'parcel_id_admin2'
if parcels[join_id_column].duplicated().any():
    raise ValueError(f'Join column {join_id_column} is not unique.')

In [ ]:
# Join count
parcels = parcels.join(
    parcels.groupby(join_id_column).size().rename('n_properties'), on=join_id_column
)

In [ ]:
parcels.sample(5).T

In [ ]:
import pandas as pd

from openplaces.io.aggregate import aggregate_rows
from openplaces.recipe import get_table_recipe

property_recipe = get_table_recipe(recipe_id, 'property')
recipe_cols = [
    c for c in property_recipe.get('columns', {}).keys() if c != join_id_column
]

In [ ]:
single_parcel_ids = set(parcels.loc[parcels['n_properties'] == 1, join_id_column])
single_mask = properties[join_id_column].isin(single_parcel_ids)

agg_single = aggregate_rows(properties[single_mask], by=join_id_column)
agg_multi = aggregate_rows(
    properties[~single_mask],
    by=join_id_column,
    # list_columns=recipe_cols,
)

agg_properties = pd.concat([agg_single, agg_multi])
parcels = parcels.join(agg_properties, on=join_id_column)

In [ ]:
parcels.sample(5).T

# Save properties

In [ ]:
save_parquet(properties, path(admin_id, 'property-openplaces-2026'))

In [ ]:
# share(
#     buildings,
#     share_path(admin_id, 'property-openplaces-2026'),
#     'share/2026/cheer',
#     delete_original=False,
# )

In [ ]:
assert False, 'This marks the end of a flattened `for` loop.'

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/.../'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Loop script

In [ ]:
CHEER_ADMIN3_IDS = 'US-NC-BE US-NC-BT US-NC-BL US-NC-BR US-NC-CM US-NC-AR US-NC-CW US-NC-CO US-NC-CR US-NC-CU US-NC-CI US-NC-DR US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL US-NC-HA US-NC-HR US-NC-HO US-NC-HD US-NC-JO US-NC-JN US-NC-LN US-NC-MR US-NC-NA US-NC-NH US-NC-NO US-NC-ON US-NC-PM US-NC-PA US-NC-PE US-NC-PQ US-NC-PI US-NC-RB US-NC-SA US-NC-SC US-NC-TY US-NC-WA US-NC-WR US-NC-WS US-NC-WY US-NC-WI'.split()

In [ ]:
for admin3_id in CHEER_ADMIN3_IDS:
    print(admin3_id)

    args_list_cheer = ['--admin_ids'] + [admin3_id] + ['--verbose']

    print(' '.join(args_list_cheer))
    test_script(*args_list_cheer)